In [3]:
#cz_type = c
# Fast ZIP code estimation using BallTree and appending to original 2023.csv
import pandas as pd
import numpy as np
from sklearn.neighbors import BallTree

# Load full 2023 dataset
full_df = pd.read_csv("C:/Users/jwan0/Downloads/Climate risk/climate data/Raw data/2023.csv", low_memory=False)

# Filter for C-type rows with lat/lon
county_coords_df = full_df[full_df['CZ_TYPE'] == 'C'].copy()
county_coords_df = county_coords_df[['STATE', 'CZ_NAME', 'BEGIN_LAT', 'BEGIN_LON']].dropna().drop_duplicates()

# Load ZIP centroid data
zip_df = pd.read_excel("C:/Users/jwan0/Downloads/uszips_2020_Ready_Signal-10.30.2024.xlsx", sheet_name="Data 2020", skiprows=30,
                       names=['zip', 'city', 'state_id', 'state_name', 'county_fips', 'county_name', 'timezone', 'latitude', 'longitude'])
zip_df['zip'] = zip_df['zip'].astype(str).str.zfill(5)
zip_df = zip_df.dropna(subset=['latitude', 'longitude'])

# Convert to radians
county_coords = np.radians(county_coords_df[['BEGIN_LAT', 'BEGIN_LON']].to_numpy())
zip_coords = np.radians(zip_df[['latitude', 'longitude']].to_numpy())

# Build BallTree and query
tree = BallTree(zip_coords, metric='haversine')
dists, indices = tree.query(county_coords, k=20)
dists *= 6371  # Convert to km

# Merge ZIPs into unique county coordinates
for i in range(20):
    county_coords_df[f'zip{i+1}'] = zip_df.iloc[indices[:, i]]['zip'].values
    county_coords_df[f'dist_km_{i+1}'] = dists[:, i]

# Compute county centroids from original full_df
county_centroids = full_df[full_df['CZ_TYPE'] == 'C'].groupby(['STATE', 'CZ_NAME']).agg(
    CENTROID_LAT=('BEGIN_LAT', 'mean'),
    CENTROID_LON=('BEGIN_LON', 'mean')
).reset_index()

# Merge both ZIP and centroid into original full_df
full_df = full_df.merge(county_coords_df, on=['STATE', 'CZ_NAME', 'BEGIN_LAT', 'BEGIN_LON'], how='left')
full_df = full_df.merge(county_centroids, on=['STATE', 'CZ_NAME'], how='left')

# Save to file
full_df.to_csv("2023_with_ZIP_and_Centroids.csv", index=False)
print("Saved to 2023_with_ZIP_and_Centroids.csv")


Saved to 2023_with_ZIP_and_Centroids.csv


In [1]:
# Fast ZIP code estimation using BallTree and appending to original 2023.csv
import pandas as pd
import numpy as np
from sklearn.neighbors import BallTree

# Load full 2023 dataset
full_df = pd.read_csv("C:/Users/jwan0/Downloads/Climate risk/climate data/2023.csv", low_memory=False)

# Filter for C-type rows with lat/lon
county_coords_df = full_df[full_df['CZ_TYPE'] == 'C'].copy()
county_coords_df = county_coords_df[['STATE', 'CZ_NAME', 'BEGIN_LAT', 'BEGIN_LON']].dropna().drop_duplicates()

# Load ZIP centroid data
zip_df = pd.read_excel("C:/Users/jwan0/Downloads/uszips_2020_Ready_Signal-10.30.2024.xlsx", sheet_name="Data 2020", skiprows=30,
                       names=['zip', 'city', 'state_id', 'state_name', 'county_fips', 'county_name', 'timezone', 'latitude', 'longitude'])
zip_df['zip'] = zip_df['zip'].astype(str).str.zfill(5)
zip_df = zip_df.dropna(subset=['latitude', 'longitude'])

# Convert to radians for BallTree
county_coords = np.radians(county_coords_df[['BEGIN_LAT', 'BEGIN_LON']].to_numpy())
zip_coords = np.radians(zip_df[['latitude', 'longitude']].to_numpy())

# Build BallTree and query for 3 closest ZIPs
tree = BallTree(zip_coords, metric='haversine')
dists, indices = tree.query(county_coords, k=3)
dists *= 6371  # Convert to km

# Merge ZIPs into unique county coordinates
for i in range(3):
    county_coords_df[f'zip{i+1}'] = zip_df.iloc[indices[:, i]]['zip'].values
    county_coords_df[f'dist_km_{i+1}'] = dists[:, i]

# Compute county centroids from original full_df
county_centroids = full_df[full_df['CZ_TYPE'] == 'C'].groupby(['STATE', 'CZ_NAME']).agg(
    CENTROID_LAT=('BEGIN_LAT', 'mean'),
    CENTROID_LON=('BEGIN_LON', 'mean')
).reset_index()

# Estimate ZIP for county centroid
centroid_coords = np.radians(county_centroids[['CENTROID_LAT', 'CENTROID_LON']].to_numpy())
dists_centroid, indices_centroid = tree.query(centroid_coords, k=1)
county_centroids['CENTROID_ZIP'] = zip_df.iloc[indices_centroid[:, 0]]['zip'].values

# Merge all results into full_df
full_df = full_df.merge(county_coords_df, on=['STATE', 'CZ_NAME', 'BEGIN_LAT', 'BEGIN_LON'], how='left')
full_df = full_df.merge(county_centroids, on=['STATE', 'CZ_NAME'], how='left')

# Save to file
full_df.to_csv("2023_with_ZIP_and_Centroids.csv", index=False)
print("Saved to 2023_with_ZIP_and_Centroids.csv")

Saved to 2023_with_ZIP_and_Centroids.csv


In [20]:
#cz_type = z

import pandas as pd

# CSV 파일 불러오기
df = pd.read_csv("C:/Users/jwan0\Downloads/Climate risk/climate data/Raw data/2007.csv", low_memory=False)

# Z 타입 필터링
cz_z_df = df[df['CZ_TYPE'] == 'Z'].copy()

# 수치형으로 변환
cz_z_df['DAMAGE_PROPERTY'] = pd.to_numeric(cz_z_df['DAMAGE_PROPERTY'], errors='coerce')
cz_z_df['DAMAGE_CROPS'] = pd.to_numeric(cz_z_df['DAMAGE_CROPS'], errors='coerce')

# 집계
pivot_df = cz_z_df.groupby(['STATE', 'EVENT_TYPE']).agg(
    event_count=('EVENT_ID', 'count'),
    total_damage_property=('DAMAGE_PROPERTY', 'sum'),
    total_damage_crops=('DAMAGE_CROPS', 'sum')
).reset_index()

# 열 확장 (pivot wider)
pivot_wide = pivot_df.pivot(index='STATE', columns='EVENT_TYPE')
pivot_wide.columns = ['_'.join(col).strip().lower().replace(' ', '_') for col in pivot_wide.columns.values]
pivot_wide = pivot_wide.reset_index()

# NA를 0으로 채우기
pivot_wide_filled = pivot_wide.fillna(0)

# 저장
pivot_wide_filled.to_csv("2007 cz_type_Z_state_summary.csv", index=False)


<>:6: SyntaxWarning: invalid escape sequence '\D'
<>:6: SyntaxWarning: invalid escape sequence '\D'
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_30920\1452785258.py:6: SyntaxWarning: invalid escape sequence '\D'
  df = pd.read_csv("C:/Users/jwan0\Downloads/Climate risk/climate data/Raw data/2007.csv", low_memory=False)
